In [1]:
import sys
# TO CHANGE
BASEDIR = "/home/dzigen/Desktop/PersonalAI/Personal-AI"
sys.path.insert(0, BASEDIR)

In [2]:
from src.graph_main import RemoteKnowledgeGraph, RemoteKnowledgeGraphConfig
from src.neo4j_functions import Neo4jConnectionConfig
from src.embedding_functions import EmbedderModelConfig, EmbeddingsDatabaseConnectionConfig, VectorDBConnectionConfig

from src.qa_pipeline import QAPipelineConfig
from src.qa_pipeline.query_parser import QueryLLMParserConfig
from src.qa_pipeline.knowledge_comparator import KnowledgeComparatorConfig

from src.qa_pipeline.knowledge_retriever import KnowledgeRetrieverConfig
from src.qa_pipeline.knowledge_retriever.AStarTripletsRetriever import AStarGraphSearchConfig
from src.qa_pipeline.knowledge_retriever.BFSTripletsRetriever import BFSSearchConfig
from src.qa_pipeline.knowledge_retriever.cache import KeyValueStoreConfig
from src.qa_pipeline.knowledge_retriever.cache.configs import DEFAULT_KVDB_CONFIGS

from src.qa_pipeline.answer_generator import QALLMGeneratorConfig

from src.utils import Logger

/home/dzigen/Desktop/PersonalAI/Personal-AI/pai_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
main_config = RemoteKnowledgeGraphConfig(
    graph_db_config=Neo4jConnectionConfig(uri='bolt://localhost:7687', user='neo4j', pwd='password', db_name='testing3'),
    embedder_db_config=EmbeddingsDatabaseConnectionConfig(
        nodes_db_config=VectorDBConnectionConfig(
            path='../data/graph_structures/vectorized_nodes/testing3', db_name='vectorized_nodes', is_exist=False, need_to_clear=True
        ),
        triplets_db_config=VectorDBConnectionConfig(
            path='../data/graph_structures/vectorized_triplets/testing3', db_name='vectorized_triplets', is_exist=False, need_to_clear=True
        ),
        embedder_config=EmbedderModelConfig(
            model_name_or_path='../models/intfloat/multilingual-e5-small'
        )
    ),
    qa_pipeline_config=QAPipelineConfig(
        query_parser_config=QueryLLMParserConfig(),
        knowledge_comparator_config=KnowledgeComparatorConfig(),
        knowledge_retriever_config=KnowledgeRetrieverConfig(
            retriever_method='astar',
            retriever_config=AStarGraphSearchConfig(),
            cache_config=KeyValueStoreConfig(
                db_vendor='inmemory',
                db_config=DEFAULT_KVDB_CONFIGS['inmemory']
            )
        )
    ),
    log=Logger('main_debug')
)

In [4]:
rkg_main = RemoteKnowledgeGraph(main_config)

No sentence-transformers model found with name ../models/intfloat/multilingual-e5-small. Creating a new one with mean pooling.


In [5]:
rkg_main.update_memory(["Every hunter wants to know: where the pheasant sits"])

  0%|          | 0/9 [00:00<?, ?it/s]Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownLabelWarning} {category: UNRECOGNIZED} {title: The provided label is not in the database.} {description: One of the labels in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing label name is: object)} {position: line: 1, column: 13, offset: 12} for query: 'MATCH (subj:object) WHERE subj.name = "hunter" RETURN elementID(subj) as node_id'
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database} {description: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your 

all/unique_triplets - 9/9
all/unique_nodes - 18/6


100%|██████████| 1/1 [00:00<00:00,  1.12it/s]

all/unique_triplets - 9/3
all/unique_nodes - 18/6


In [6]:
rkg_main.answer_question("What does every hunter want to know?")

Number of requested results 20 is greater than number of elements in index 6, updating n_results = 6
Number of requested results 20 is greater than number of elements in index 6, updating n_results = 6
Number of requested results 50 is greater than number of elements in index 3, updating n_results = 3


'It depends on the context and individual hunters, but generally, they would want to know information about their prey, such as location, behavior, and patterns.'